In [ ]:
from azureml.core import Workspace, Dataset
from azureml.datadrift import DataDriftDetector

from datetime import datetime, timedelta

In [ ]:
ws     = Workspace.from_config()
dstore = ws.get_default_datastore()

In [ ]:
dstore.upload('weather-data', 'florida-weather', overwrite=True)

In [ ]:
dset   = Dataset.Tabular.from_parquet_files(dstore.path('florida-weather/**/data.parquet'))

In [ ]:
target = dset.with_timestamp_columns('datetime')
target = target.register(ws, 'target')

In [ ]:
target = Dataset.get_by_name(ws, 'target')

In [ ]:
baseline = target.time_before(datetime(2011, 1, 1))

In [ ]:
features = ['latitude', 'longitude', 'elevation', 'windAngle', 'windSpeed', 'temperature', 'snowDepth', 'stationName', 'countryOrRegion']

In [ ]:
monitor = DataDriftDetector.create_from_datasets(ws, 'weatherMonitor2', baseline, target, 
                                                      compute_target_name='cpu-cluster', 
                                                      frequency='Month', 
                                                      feature_list=None, 
                                                      drift_threshold=None, 
                                                      latency=0)

In [ ]:
monitor = DataDriftDetector.get_by_name(ws, 'weatherMonitor2')

monitor = monitor.update(drift_threshold=.6)
monitor = monitor.update(feature_list=features)

In [ ]:
for year in range(2010, 2020): monitor.backfill(datetime(year, 1, 1), datetime(year+1, 1, 1))

In [ ]:
help(monitor)